# Full pipeline - FastSpeech version
text → FastSpeech-style Transformer → Diffusion → GAN vocoder → audio.

In [1]:
import sys, torch
ROOT = r"C:\Users\INTEL\Downloads\voice_cloning_v2_1\voice_cloning_v2_1"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select the Python 3.11.9 kernel")

GPU: NVIDIA GeForce RTX 4070


In [2]:
from IPython.display import Audio, display
from vc_lib.infer import VoicePipeline, evaluate
from vc_lib.common import SR

pipe = VoicePipeline(ROOT, diffusion_steps=50, cfg=1.0)
fast = VoicePipeline(ROOT, use_diffusion=False)
spk = pipe.meta['train_speakers'][0]; unseen = pipe.meta['holdout'][0]
text = 'Hello! Thanks for calling. How can I help you today?'

Text-to-mel model: fastspeech | diffusion: on
Text-to-mel model: fastspeech | diffusion: off


## Listen: with and without diffusion

In [3]:
display(Audio(pipe.synthesize(text, speaker=spk), rate=SR))
display(Audio(fast.synthesize(text, speaker=spk), rate=SR))

c:\Users\INTEL\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Speaking rate (pace < 1 = faster, > 1 = slower)

In [4]:
display(Audio(fast.synthesize(text, speaker=spk, pace=0.85), rate=SR))
display(Audio(fast.synthesize(text, speaker=spk, pace=1.15), rate=SR))

## Unseen speaker

In [5]:
display(Audio(fast.synthesize(text, speaker=unseen), rate=SR))

## Your own voice
Put the path of a WAV file (5-10 s, quiet room) below.

In [6]:
ref = r"C:\Users\INTEL\Downloads\Recording.wav"
emb = fast.embed_reference([ref])
display(Audio(fast.synthesize(text, speaker_emb=emb), rate=SR))

## Evaluate (WER: lower is better; similarity: higher is better)

In [7]:
sentences = ['Please call Stella.', 'The weather is nice today.', 'I will send the report tomorrow morning.',
             'Can you repeat that, please?', 'Your order has been shipped.']
print('--- FastSpeech, no diffusion ---'); evaluate(fast, sentences, speaker=spk)
print('--- FastSpeech + diffusion ---'); evaluate(pipe, sentences, speaker=spk)
print('--- unseen speaker ---'); evaluate(fast, sentences, speaker=unseen)

--- FastSpeech, no diffusion ---
WER 0.33 | similarity 0.55 | 'Please call Stella.' -> 'Please call st-'
WER 0.00 | similarity 0.86 | 'The weather is nice today.' -> 'The weather is nice today.'
WER 0.00 | similarity 0.85 | 'I will send the report tomorrow morning.' -> 'I will send the report tomorrow morning.'
WER 0.00 | similarity 0.73 | 'Can you repeat that, please?' -> 'Can you repeat that, please?'
WER 0.00 | similarity 0.83 | 'Your order has been shipped.' -> 'Your order has been shipped.'

Mean WER: 0.067 (lower is better)   Mean speaker similarity: 0.764
--- FastSpeech + diffusion ---
WER 1.33 | similarity 0.54 | 'Please call Stella.' -> 'P.S. COOL STA-'
WER 0.80 | similarity 0.81 | 'The weather is nice today.' -> "The weather is what it's to get."
WER 0.43 | similarity 0.81 | 'I will send the report tomorrow morning.' -> "I'll send an report tomorrow morning."
WER 0.00 | similarity 0.57 | 'Can you repeat that, please?' -> 'Can you repeat that, please?'
WER 1.00 | similarity 0.

([0.3333333333333333, 0.0, 0.0, 0.0, 0.0],
 [0.4633965492248535,
  0.5403584837913513,
  0.6572751998901367,
  0.4703759551048279,
  0.636787474155426])

## Baseline comparison (for your report)
The original fixed-slot Transformer, same vocoder.

In [8]:
base = VoicePipeline(ROOT, use_diffusion=False, model='baseline')
evaluate(base, sentences, speaker=spk)

Text-to-mel model: baseline | diffusion: off
WER 1.00 | similarity 0.41 | 'Please call Stella.' -> 'Peace.'
WER 1.00 | similarity 0.75 | 'The weather is nice today.' -> 'Pffftttttt'
WER 1.00 | similarity 0.64 | 'I will send the report tomorrow morning.' -> ''
WER 0.80 | similarity 0.55 | 'Can you repeat that, please?' -> 'You'
WER 1.00 | similarity 0.78 | 'Your order has been shipped.' -> 'Ehhhhhhhhhh...  S.A.P.'

Mean WER: 0.960 (lower is better)   Mean speaker similarity: 0.629


([1.0, 1.0, 1.0, 0.8, 1.0],
 [0.4148932695388794,
  0.7506838440895081,
  0.6437431573867798,
  0.5532375574111938,
  0.7835521697998047])

## Save audio

In [9]:
import soundfile as sf, os
sf.write(os.path.join(ROOT, 'demo_output.wav'), fast.synthesize(text, speaker=spk), SR)